# DR-PINN on Google Colab · Milestones 1–8

**Physics-Informed Neural Networks for Retinal Hemodynamic Stress Estimation and Early Risk Mapping in Diabetic Retinopathy** — research prototype.

> ⚠️ Research prototype, not a medical device. Model-derived risk estimates are experimental and need independent clinical validation.

**How to use**
1. Download `dr-pinn-M11.2.zip` and this notebook from the chat.
2. **Runtime → Run all.** Allow Google Drive access. If Drive holds an older milestone (or nothing), you'll be asked to choose the zip.
3. Everything lives in **MyDrive/dr-pinn**, because Colab deletes its own disk when a session ends. Your `data/`, `checkpoints/` and `reports/` are never overwritten.

**Runtime:** steps 12 (U-Net) and 15 (training the triage model) need **Runtime → Change runtime type → T4 GPU**. Everything else runs on CPU, including re-evaluating a trained triage model and the dashboard (step 16).

## 1 · Connect Google Drive

In [ ]:
import os, re, sys, shutil, zipfile, subprocess, tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/dr-pinn")
else:  # plain Jupyter on your own machine: run from dr-pinn/notebooks/
    PROJECT = Path(os.environ.get("DRPINN_PROJECT", Path.cwd().parent)).resolve()
DRIVE_ROOT = PROJECT.parent
print("Project folder:", PROJECT)

## 2 · Make sure Drive has the matching code
Installs or upgrades automatically. Tick `FORCE_REINSTALL` only if the code in Drive got damaged.

In [ ]:
# @title Install / upgrade { display-mode: "form" }
NOTEBOOK_VERSION = "0.8.2"  # this notebook belongs to code version 0.8.2
FORCE_REINSTALL = False  # @param {type:"boolean"}

KEEP = {"data", "checkpoints", "reports"}  # your results and datasets, never overwritten
_VER = re.compile(r'__version__\s*=\s*"([^"]+)"')

def version_in_folder(folder):
    init = folder / "src" / "drpinn" / "__init__.py"
    m = _VER.search(init.read_text()) if init.exists() else None
    return m.group(1) if m else None

def version_in_zip(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as z:
            m = _VER.search(z.read("dr-pinn/src/drpinn/__init__.py").decode())
    except (KeyError, zipfile.BadZipFile):
        return None
    return m.group(1) if m else None

def matching_zip():
    for zp in sorted(DRIVE_ROOT.glob("dr-pinn*.zip"), key=lambda p: p.stat().st_mtime, reverse=True):
        if version_in_zip(zp) == NOTEBOOK_VERSION:
            return zp
    if not IN_COLAB:
        raise FileNotFoundError(f"no dr-pinn zip with version {NOTEBOOK_VERSION} in {DRIVE_ROOT}")
    from google.colab import files
    print(f"Choose dr-pinn-M11.2.zip (code version {NOTEBOOK_VERSION}) from your downloads…")
    zp = Path.cwd() / next(iter(files.upload()))
    found = version_in_zip(zp)
    if found != NOTEBOOK_VERSION:
        raise ValueError(f"{zp.name} has code version {found}, this notebook needs {NOTEBOOK_VERSION} — upload dr-pinn-M11.2.zip")
    return zp

def install_zip(zip_path):
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(tmp)
        src = Path(tmp) / "dr-pinn"
        PROJECT.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            dst = PROJECT / item.name
            if item.name in KEEP:  # add new files only
                if item.is_dir():
                    shutil.copytree(item, dst, dirs_exist_ok=True,
                                    copy_function=lambda s, d: None if Path(d).exists() else shutil.copy2(s, d))
                continue
            if dst.is_dir():
                shutil.rmtree(dst)
            elif dst.exists():
                dst.unlink()
            shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)
    print(f"Installed {zip_path.name} → {PROJECT}")

installed = version_in_folder(PROJECT)
if FORCE_REINSTALL or installed != NOTEBOOK_VERSION:
    print(f"Drive has code version {installed or 'none'}; this notebook needs {NOTEBOOK_VERSION}.")
    install_zip(matching_zip())
else:
    print(f"Code version {installed} already in Drive — nothing to do.")

## 3 · Check the machine

In [ ]:
import platform, numpy, scipy
print(f"Python {platform.python_version()} | numpy {numpy.__version__} | scipy {scipy.__version__} | CPUs {os.cpu_count()}")
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime — fine for now)"
    print(f"PyTorch {torch.__version__} | GPU: {gpu}")
except ImportError:
    print("PyTorch not installed (not needed until Milestone 5)")

## 4 · Install requirements and load the package

In [ ]:
# Installs only what is missing; it does not upgrade Colab's own numpy/scipy.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
                   capture_output=True, text=True)
print(r.stdout.strip() or "requirements already satisfied")
if r.returncode:
    print(r.stderr); raise RuntimeError("pip install failed")

SRC = str(PROJECT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for m in [m for m in sys.modules if m == "drpinn" or m.startswith("drpinn.")]:
    del sys.modules[m]  # pick up new code after an upgrade without restarting

import drpinn
print("drpinn", drpinn.__version__, "loaded from", Path(drpinn.__file__).parent)
print(drpinn.DISCLAIMER)

# Everything later cells use, loaded once here, so any cell works after steps 1-4
# (e.g. after Colab restarts the runtime).
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as ShowImage
from drpinn.hemodynamics import (BoundaryConditions, CarreauYasudaViscosity, murray_tree,
                                 pries_relative_viscosity, random_tree, solve_network)
from drpinn.hemodynamics.units import cp_to_pa_s, m3_s_to_ul_per_min, mmhg_to_pa
from drpinn.datasets import find_drive_samples
DRIVE_DIR = PROJECT / "data" / "DRIVE"
DRIVE_READY = DRIVE_DIR.exists() and len(find_drive_samples(DRIVE_DIR)) > 0

def run(*args):
    """Run a project command from the project folder and show its output."""
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTHONPATH": SRC}
    r = subprocess.run([sys.executable, *map(str, args)], cwd=PROJECT, env=env,
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr)
        raise RuntimeError(f"command failed with exit code {r.returncode}")

def run_live(*args):
    """Like run(), but prints output as it arrives (for long steps)."""
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTHONPATH": SRC, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen([sys.executable, *map(str, args)], cwd=PROJECT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed with exit code {p.returncode}")

def unzip_all(folder):
    """Extract every zip in `folder` into a sibling folder named after it (once)."""
    for z in sorted(Path(folder).glob("*.zip")):
        target = z.with_suffix("")
        if not target.exists():
            print("Unzipping", z.name, "…", flush=True)
            with zipfile.ZipFile(z) as f:
                f.extractall(target)

## 5 · Run the test suite
Expected: **235 passed**. If anything fails, copy the whole output into the chat.

In [ ]:
run("-m", "pytest", "-p", "no:cacheprovider")

## 6 · Synthetic demo: a narrowed segment and a blood-pressure sweep

In [ ]:
run("scripts/demo_synthetic.py")

## 7 · Solver speed on Colab's hardware
Saved to `reports/benchmarks/solver_latency_colab.json` in Drive.

In [ ]:
run("scripts/benchmark_solver.py", "--repeats", "20", "--mc-samples", "1000",
    "--out", PROJECT / "reports/benchmarks/solver_latency_colab.json")

## 8 · Narrowing explorer
Move the sliders and re-run. **A:** a Murray's-law tree has the same wall shear stress everywhere. **C:** narrowing one segment gives high shear there (red) and low shear downstream (blue). **Change `INLET_MMHG`:** the numbers in the titles change, the colours and ratios don't — blood pressure alone cannot reorder segments.

*Synthetic network, illustrative parameters — not patient data.*

In [ ]:
# @title Narrowing explorer { display-mode: "form" }
GENERATIONS = 7          # @param {type:"slider", min:3, max:10, step:1}
NARROWED_GENERATION = 2  # @param {type:"slider", min:1, max:8, step:1}
RADIUS_FRACTION = 0.5    # @param {type:"slider", min:0.2, max:1.0, step:0.05}
INLET_MMHG = 40          # @param {type:"slider", min:20, max:80, step:1}
OUTLET_MMHG = 15         # @param {type:"slider", min:0, max:30, step:1}
VISCOSITY_CP = 3.5       # @param {type:"number"}

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm, TwoSlopeNorm
from drpinn.hemodynamics import BoundaryConditions, murray_tree, solve_network
from drpinn.hemodynamics.units import cp_to_pa_s, m3_s_to_ul_per_min, mmhg_to_pa

if not 1 <= NARROWED_GENERATION < GENERATIONS:
    raise ValueError("NARROWED_GENERATION must be between 1 and GENERATIONS - 1")
if OUTLET_MMHG >= INLET_MMHG:
    raise ValueError("INLET_MMHG must be above OUTLET_MMHG")

syn = murray_tree(GENERATIONS, root_radius_m=60e-6, length_to_radius=20)
bc = BoundaryConditions.inlet_outlet_pressure(
    syn.inlet_nodes, mmhg_to_pa(INLET_MMHG), syn.outlet_nodes, mmhg_to_pa(OUTLET_MMHG))
mu = cp_to_pa_s(VISCOSITY_CP)
base = solve_network(syn.network, bc, mu)

e = int(np.flatnonzero(syn.edge_generation == NARROWED_GENERATION)[0])
radius = syn.network.radius_m.copy()
radius[e] *= RADIUS_FRACTION
narrowed = solve_network(syn.network.with_radius(radius), bc, mu)
ratio = narrowed.wss_pa / base.wss_pa
down = syn.descendants(e)

segs = (syn.network.node_xy * 1e3)[syn.network.edges]
width_of = lambda r: 0.4 + 4.5 * r / syn.network.radius_m.max()  # line width ~ vessel radius
wss_all = np.concatenate([base.wss_pa, narrowed.wss_pa])
wss_norm = LogNorm(vmin=wss_all.min() * 0.95, vmax=wss_all.max() * 1.05)
ratio_norm = TwoSlopeNorm(vcenter=1.0, vmin=min(ratio.min(), 0.99), vmax=max(ratio.max(), 1.01))

fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.0))
panels = [
    (base.wss_pa, syn.network.radius_m, wss_norm, "viridis", f"A · Murray tree\nWSS max {base.wss_pa.max():.1f} Pa", "WSS (Pa)"),
    (narrowed.wss_pa, radius, wss_norm, "viridis", f"B · segment {e} at {RADIUS_FRACTION:.0%} radius\nWSS max {narrowed.wss_pa.max():.1f} Pa", "WSS (Pa)"),
    (ratio, radius, ratio_norm, "coolwarm", "C · ratio B / A\nred higher, blue lower", "ratio"),
]
for ax, (vals, r_draw, norm, cmap, title, label) in zip(axes, panels):
    lc = LineCollection(segs, array=vals, cmap=cmap, norm=norm, linewidths=width_of(r_draw), capstyle="round")
    ax.add_collection(lc)
    ax.plot(*segs[e].mean(axis=0), "o", ms=11, mfc="none", mec="black", mew=1.3)
    ax.autoscale(); ax.set_aspect("equal"); ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=8); ax.set_xlabel("mm", fontsize=8)
    cb = fig.colorbar(lc, ax=ax, shrink=0.75); cb.set_label(label, fontsize=8); cb.ax.tick_params(labelsize=7)
fig.suptitle(f"SYNTHETIC · P_in {INLET_MMHG} mmHg, P_out {OUTLET_MMHG} mmHg, μ {VISCOSITY_CP} cP (illustrative)", fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

print(f"Narrowed segment {e}: WSS x{ratio[e]:.2f}")
if down.size:
    print(f"{down.size} downstream segments: WSS x{ratio[down].mean():.2f} on average")
print(f"Total inflow: {m3_s_to_ul_per_min(base.total_inflow_m3_s):.2f} → "
      f"{m3_s_to_ul_per_min(narrowed.total_inflow_m3_s):.2f} uL/min")
print(f"Mass-conservation error: {narrowed.max_rel_conservation_error:.1e}")

## 9 · Milestone 2 — which unknowns actually reorder the ranking?
Twenty synthetic trees (~255 segments each). Each uncertain input is perturbed 20 times and the wall-shear-stress ranking is compared with the baseline. The key number is the **share of the baseline top-10 % segments that stay in the top 10 %** — that is the "high-risk" set a report would show. Takes about 15 s.

*Synthetic trees, illustrative perturbation sizes. These decide what M7 must sample; the sizes will be replaced by measured segmentation error and literature ranges.*

In [ ]:
run("scripts/m2_rank_sensitivity.py", "--trees", "20", "--samples", "20")

In [ ]:
import json
import pandas as pd
from matplotlib.patches import Patch

res = json.loads((PROJECT / "reports/experiments/m2_rank_sensitivity.json").read_text())
df = pd.DataFrame(res["results"])
colors = {"control (proved invariant)": "#9e9e9e", "boundary conditions": "#d95f02",
          "geometry error": "#1b9e77", "rheology": "#7570b3"}

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(df))[::-1]
ax.barh(y, df.top10_overlap_mean, color=df.group.map(colors))
ax.plot(df.top10_overlap_p05, y, "k|", ms=12, mew=2)
ax.set_yticks(y, [f"{p}  ({d})" for p, d in zip(df.perturbation, df.description)], fontsize=8)
ax.set_xlim(0, 1.02); ax.axvline(1, color="k", lw=0.5)
ax.set_xlabel("Share of baseline top-10% segments still in the top 10%   (bar = mean, | = worst 5%)")
fig.legend(handles=[Patch(color=c, label=g) for g, c in colors.items()], loc="lower center", ncol=4, fontsize=8)
ax.set_title("SYNTHETIC · what reorders the wall-shear-stress ranking", fontsize=10)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

df[["perturbation", "description", "spearman_mean", "top10_overlap_mean", "top10_overlap_p05"]].round(3)

**How to read it**
- **Grey (controls):** blood pressure, viscosity, image scale and a uniform radius bias change nothing — exactly as proved in M1.
- **Orange (outlet conditions):** the biggest effect. How the unseen downstream bed is modelled swaps out about half of the top-10 % segments, and a photo cannot tell us which assumption is right. These must be sampled in M7, and the report must say so.
- **Green (segmentation error):** radius errors of 10–20 % per segment swap out 35–60 % of the top set. This sets an accuracy target for M3/M5: radius needs to be measured to within roughly 5 %, which means sub-pixel vessel-width estimation.
- **Purple (rheology):** Fåhræus–Lindqvist matters a little; shear-thinning is negligible at these shear rates (see the next plot), so it can be dropped with this evidence.

In [ ]:
from drpinn.hemodynamics import CarreauYasudaViscosity, pries_relative_viscosity, random_tree

tree = random_tree(8, seed=0)
sol = solve_network(tree.network, BoundaryConditions.inlet_outlet_pressure(
    tree.inlet_nodes, mmhg_to_pa(40), tree.outlet_nodes, mmhg_to_pa(15)), 3.5e-3)
diam_um = 2e6 * tree.network.radius_m

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))
d = np.logspace(np.log10(5), np.log10(1000), 300)
for hd in (0.30, 0.45, 0.55):
    a1.plot(d, pries_relative_viscosity(d, hd), label=f"Hd {hd}")
a1.axvspan(diam_um.min(), diam_um.max(), color="orange", alpha=0.2, label="diameters in a synthetic tree")
a1.set_xscale("log"); a1.set_xlabel("vessel diameter (µm)"); a1.set_ylabel("viscosity relative to plasma")
a1.set_title("Fåhræus–Lindqvist (Pries et al. 1992, in vitro)", fontsize=10); a1.legend(fontsize=8)

g = np.logspace(-2, 5, 300)
a2.loglog(g, CarreauYasudaViscosity().viscosity_at(g) * 1e3, label="Carreau–Yasuda (Cho & Kensey)")
a2.axvspan(sol.wall_shear_rate_s.min(), sol.wall_shear_rate_s.max(), color="orange", alpha=0.2,
           label="wall shear rates in a synthetic tree")
a2.set_xlabel("shear rate (1/s)"); a2.set_ylabel("viscosity (mPa·s)")
a2.set_title("Shear-thinning: the tree sits on the flat part", fontsize=10); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10 · Milestone 3 — from a vessel mask to a vessel graph
The solver needs a graph: junctions, segments, and each segment's length and radius. M2 showed radius must be measured to within ~5 %.

### 10a · How accurately can radius be measured? (synthetic tubes of known radius, ~20 s)
Three methods: the **distance transform** the PDF proposes, and the new **area method** on a binary mask and on a soft (probability) mask.

In [ ]:
run("scripts/m3_width_accuracy.py")
wa = pd.DataFrame(json.loads((PROJECT / "reports/experiments/m3_width_accuracy.json").read_text())["results"])
fig, ax = plt.subplots(figsize=(8, 4))
for col, label in (("edt_binary_median_abs", "distance transform (PDF)"),
                   ("area_binary_median_abs", "area method, binary mask"),
                   ("area_soft_median_abs", "area method, soft mask")):
    ax.plot(wa.radius_px, 100 * wa[col], "o-", label=label)
ax.axhline(5, color="red", ls="--", lw=1, label="M2 target: 5 %")
ax.set_yscale("log"); ax.set_xlabel("true vessel radius (pixels)"); ax.set_ylabel("median radius error (%)")
ax.set_title("SYNTHETIC · radius error by method", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 10b · Put DRIVE into your Drive (one time)
1. Go to **drive.grand-challenge.org**, sign in (free account), open **Download** in the challenge menu and download the **training** set. Only the 20 training images include expert vessel masks; test-set masks are withheld by the organisers.
2. If the official link is down (this has happened), a Kaggle copy of DRIVE works — any folder layout is fine.
3. In Google Drive, put the zip (or the unzipped folder) inside **MyDrive/dr-pinn/data/DRIVE/**.
4. Run the next cell. It unzips if needed and counts the images it found.

Cite in your thesis: Staal et al., *Ridge-based vessel segmentation in color images of the retina*, IEEE TMI 23(4):501–509, 2004.

In [ ]:
from drpinn.datasets import find_drive_samples

DRIVE_DIR = PROJECT / "data" / "DRIVE"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for z in DRIVE_DIR.glob("*.zip"):
    target = DRIVE_DIR / z.stem
    if not target.exists():
        print("Unzipping", z.name, "…")
        with zipfile.ZipFile(z) as f:
            f.extractall(target)
samples = find_drive_samples(DRIVE_DIR)
DRIVE_READY = len(samples) > 0
if DRIVE_READY:
    print(f"Found {len(samples)} DRIVE images with expert vessel masks "
          f"({sum(s.fov_path is not None for s in samples)} with field-of-view masks).")
else:
    print("No DRIVE vessel masks found yet in", DRIVE_DIR, "— follow 10b, then re-run this cell and 10c.")

### 10c · Extract vessel graphs from all DRIVE expert masks
Uses the expert masks, so this tests the graph step alone (segmentation comes in M5). In the overlays: lines are coloured by radius, orange dots are junctions, red circles are possible artery/vein crossings left unresolved, and white crosses are vessels cut by the edge of the photo.

In [ ]:
from IPython.display import Image as ShowImage, display

if DRIVE_READY:
    run("scripts/m3_drive_graphs.py", "--drive-dir", DRIVE_DIR, "--out", PROJECT / "reports/m3_drive", "--overlays", "3")
    stats = pd.read_csv(PROJECT / "reports/m3_drive/drive_graph_stats.csv", dtype={"image_id": str})
    display(stats[["image_id", "segments", "junctions", "tips", "truncated_tips", "crossings_resolved",
                   "possible_crossings", "components", "radius_px_median", "edt_minus_area_median_px"]])
    for png in sorted((PROJECT / "reports/m3_drive/overlays").glob("*.png"))[:3]:
        display(ShowImage(filename=str(png)))
else:
    print("Skipped: DRIVE not found (see 10b).")

**How to read it**
- **Width:** on thin vessels (1–2 px radius, most of DRIVE) the distance transform is off by 20–50 %; the area method stays within ~3 % on binary masks and ~0.3 % on soft masks. So in M5 the U-Net's **probability map** should be kept for width measurement, not just its thresholded mask.
- **Crossings:** clean crossings (25–90°) are split correctly; crossings that coincide with a branch point are only flagged. On synthetic fundus-like images, 2 of 9 true crossings were split and most others flagged. `components` near 1 means arteries and veins are still joined through unresolved crossings — the physics must not be run on such graphs until arteries and veins are separated (added to M5).
- **Cut-off vessels** at the photo edge are flagged so they are never treated as real vessel endings.

## 11 · Milestone 4 — the datasets
| Dataset | What it gives this project | Size | Where it goes in Drive |
|---|---|---|---|
| **IDRiD** | 516 graded images (DR 0–4, macular-oedema risk); 81 with expert lesion masks + optic disc; disc and fovea centres | 1.0 GB | `dr-pinn/data/IDRiD/` |
| **APTOS 2019** | 3,662 graded images (DR 0–4) for the triage model | 10.2 GB on Kaggle → ~0.5 GB working copy | `dr-pinn/data/APTOS2019/` (made by 11b) |
| **RITE** | artery/vein labels for the 40 DRIVE images (needed in M5) | small | `dr-pinn/data/RITE/` |
| DRIVE | already set up in step 10 | | `dr-pinn/data/DRIVE/` |

Do whichever you have now; the check in 11d works with any subset and tells you what is missing.

### 11a · IDRiD
1. Sign in at **IEEE DataPort** and open *Indian Diabetic Retinopathy Image Dataset (IDRiD)* (DOI 10.21227/H25W98). Licence: CC BY 4.0.
2. Download the three zips: **A. Segmentation**, **B. Disease Grading**, **C. Localization**.
3. In Google Drive create **dr-pinn/data/IDRiD/** and upload the three zips there (no need to unzip).
4. Run the cell below.

Cite: Porwal et al., *Indian Diabetic Retinopathy Image Dataset (IDRiD)*, IEEE DataPort, 2018.

In [ ]:
from drpinn.datasets.idrid import find_idrid

IDRID_DIR = PROJECT / "data" / "IDRiD"
IDRID_DIR.mkdir(parents=True, exist_ok=True)
unzip_all(IDRID_DIR)
try:
    idr = find_idrid(IDRID_DIR)
    print(f"IDRiD: {len(idr.grading)} graded images, {len(idr.segmentation)} with lesion masks, "
          f"{len(idr.od_centre)} disc centres, {len(idr.fovea_centre)} fovea centres")
    for p in idr.problems[:10]:
        print("  problem:", p)
except FileNotFoundError as e:
    print(e)

### 11b · APTOS 2019 (downloaded straight into Colab, ~20–40 min the first time)
APTOS is 10.2 GB — too big to keep in a free Drive. This cell downloads the competition data to Colab's temporary disk, unpacks only the 3,662 labelled training images, then saves a cropped copy (longest side 1024 px, ~0.5 GB) to Drive. If Colab disconnects, re-run: finished steps are skipped.

One-time setup (Kaggle changed its login system in 2025–26; these steps match the current site):
1. Sign in to **kaggle.com**, open the competition **APTOS 2019 Blindness Detection**, go to its **Rules** tab and accept the rules (read them — they govern how you may use the data). Kaggle may first ask you to verify your phone number.
2. Go to **kaggle.com/settings** → section **API** → **Generate New Token**, and copy the token it shows.
3. In Colab, click the **key icon** (Secrets) in the left bar → **Add new secret**: name `KAGGLE_API_TOKEN`, value = the token; switch on **Notebook access**. (An older `kaggle.json` also works as two secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`.) Never paste the token into a cell or into Drive.
4. Run the cell below and keep the tab open until it finishes. It checks your login first, before starting the 10 GB download.

In [ ]:
APTOS_DIR = PROJECT / "data" / "APTOS2019"
RAW = Path(os.environ.get("APTOS_RAW", "/content/aptos_raw"))   # Colab's temporary disk
COMP = "aptos2019-blindness-detection"
N_TRAIN, NEED_GB = 3662, 25

def n_files(folder, pattern):
    return len(list(folder.glob(pattern))) if folder.exists() else 0

def kaggle_login():
    # Reads Kaggle credentials from Colab Secrets; values are never printed.
    # New style: secret KAGGLE_API_TOKEN ("Generate New Token" on kaggle.com/settings).
    # Old style: KAGGLE_USERNAME + KAGGLE_KEY from a kaggle.json file.
    from google.colab import userdata
    def secret(name):
        try:
            v = userdata.get(name)
        except Exception:
            return None
        v = (v or "").strip().strip(chr(34) + chr(39)).strip()
        return v or None
    token, user, key = secret("KAGGLE_API_TOKEN"), secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if key and key.startswith("{"):            # the whole kaggle.json was pasted
        try:
            j = json.loads(key)
            user, key = user or j.get("username"), j.get("key")
        except ValueError:
            pass
    legacy_key = bool(key and re.fullmatch(r"[0-9a-f]{32}", key))
    if key and not legacy_key and not token:   # a new-style token pasted into KAGGLE_KEY
        token = key
    for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ.pop(k, None)
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        return f"API token from Colab Secrets ({len(token)} characters)"
    if user and legacy_key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
        return f"legacy kaggle.json key for user {user}"
    return None

def kaggle_problem(msg):
    if "403" in msg or "Forbidden" in msg:
        return ("Kaggle refused access to the competition data. Open kaggle.com/competitions/" + COMP +
                "/rules and accept the rules (Kaggle may first ask you to verify your phone), then re-run.")
    if any(s in msg for s in ("401", "Unauthorized", "Generate New Token", "auth login")):
        return ("Kaggle did not accept your credentials. On kaggle.com/settings click 'Generate New Token' "
                "under API, save the token as a Colab secret named KAGGLE_API_TOKEN (notebook access on), re-run.")
    return "see the Kaggle message above and send it to me."

def kaggle(*args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def extract_train(zip_path):
    # Unpack only train.csv and train_images/ (skips 1.6 GB of unlabelled test images).
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if not m.endswith("/") and
                   (Path(m).name == "train.csv" or "train_images/" in m)]
        print(f"Unpacking {len(members)} training files from {zip_path.name} …", flush=True)
        for k, m in enumerate(members, 1):
            name = Path(m).name
            out = RAW / name if name == "train.csv" else RAW / "train_images" / name
            if not out.exists():
                out.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            if k % 1000 == 0:
                print(f"  {k}/{len(members)}", flush=True)

if n_files(APTOS_DIR / "train_images", "*.jpg") >= N_TRAIN and (APTOS_DIR / "train.csv").exists():
    print(f"APTOS working copy already in Drive ({n_files(APTOS_DIR / 'train_images', '*.jpg')} images) — nothing to do.")
elif not IN_COLAB:
    print("Run this step in Colab.")
else:
    RAW.mkdir(parents=True, exist_ok=True)
    comp_zip = RAW / f"{COMP}.zip"
    raw_ready = (RAW / "train.csv").exists() and n_files(RAW / "train_images", "*.png") >= N_TRAIN
    login = None if raw_ready else kaggle_login()
    if not raw_ready and login is None:
        print("No Kaggle credentials found in Colab Secrets — follow steps 1–3 above, then re-run.")
    elif not raw_ready:
        print("Kaggle login:", login)
        if not comp_zip.exists():
            free_gb = shutil.disk_usage(RAW).free / 1e9
            if free_gb < NEED_GB:
                raise RuntimeError(f"Only {free_gb:.0f} GB free on Colab's disk; {NEED_GB} GB needed. "
                                   "Runtime → Disconnect and delete runtime, then re-run.")
            from importlib.metadata import PackageNotFoundError, version
            try:
                cli_major = int(version("kaggle").split(".")[0])
            except (PackageNotFoundError, ValueError):
                cli_major = 0
            if cli_major < 2 or shutil.which("kaggle") is None:
                print("Installing the current Kaggle command-line tool …", flush=True)
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle>=2"], check=True)
            code, msg = kaggle("competitions", "files", "-c", COMP)   # quick check before 10 GB
            if code:
                print(msg[-800:])
                raise RuntimeError("Kaggle check failed — " + kaggle_problem(msg))
            print("Kaggle accepted the login. Downloading the APTOS 2019 data (~10 GB; usually a few minutes) …", flush=True)
            code, msg = kaggle("competitions", "download", "-c", COMP, "-p", str(RAW))
            print(msg[-800:])
            if code or not comp_zip.exists():
                comp_zip.unlink(missing_ok=True)
                raise RuntimeError("Kaggle download failed — " + kaggle_problem(msg))
        try:
            extract_train(comp_zip)
        except zipfile.BadZipFile:
            comp_zip.unlink(missing_ok=True)
            raise RuntimeError("The download was incomplete (Colab probably disconnected). Re-run this cell.")
        if n_files(RAW / "train_images", "*.png") >= N_TRAIN:
            comp_zip.unlink()  # free 10 GB; the unpacked images are all we need
    if (RAW / "train.csv").exists() and n_files(RAW / "train_images", "*.png") > 0:
        print(f"{n_files(RAW / 'train_images', '*.png')} training images ready; making the 1024 px working copy in Drive …", flush=True)
        run_live("scripts/m4_make_working_copy.py", "--src", RAW, "--dst", APTOS_DIR,
                 "--max-side", "1024", "--workers", str(os.cpu_count() or 2))

### 11c · RITE (artery/vein labels — needed in Milestone 5)
1. Fill in the download-request form at **eye.medicine.uiowa.edu/rite-dataset**.
2. Upload the zip you receive to **dr-pinn/data/RITE/** and run the cell below.

Cite: Hu, Abràmoff & Garvin, *Automated separation of binary overlapping trees in low-contrast color retinal images*, MICCAI 2013.

In [ ]:
from drpinn.datasets.rite import find_rite

RITE_DIR = PROJECT / "data" / "RITE"
RITE_DIR.mkdir(parents=True, exist_ok=True)
unzip_all(RITE_DIR)
rite = find_rite(RITE_DIR)
print(f"RITE: {len(rite.records)} images with artery/vein labels" if rite.records else
      "RITE not found yet — optional until Milestone 5.")
for p in rite.problems[:10]:
    print("  problem:", p)

### 11d · Check every image, find duplicates, make leakage-free splits
Decodes every image (corrupt files are excluded), finds near-duplicate photos **within and across** datasets, and assigns train / val / test so that copies of one photo never end up on both sides. Writes `data/splits/manifest.csv` (used by every later milestone) and `reports/m4_data/integrity_report.json`. The first run reads every image (a few minutes for APTOS); re-runs use a cache.

In [ ]:
run_live("scripts/m4_prepare_datasets.py", "--data-dir", PROJECT / "data", "--out", PROJECT / "reports/m4_data")
rep = json.loads((PROJECT / "reports/m4_data/integrity_report.json").read_text())
split_rows = [{"dataset": k, "split": s, "images": sum(c.values()),
               "grades 0 / 1 / 2 / 3 / 4": " / ".join(str(c.get(str(g), 0)) for g in range(5))}
              for k, v in rep["splits"].items() for s, c in v.items()]
display(pd.DataFrame(split_rows))
print("Duplicate pairs by kind:", rep["duplicate_pairs_by_kind"], "| pairs to inspect by eye:", rep["review_pairs"])
print("Grouping:", rep["grouping"])
print("Manifest fingerprint:", rep["manifest_fingerprint"])
c = rep.get("aptos_split_vs_previous_manifest")
if c:
    print(f"APTOS vs previous manifest: {c['changed_split']} of {c['compared']} photos changed split",
          "→ a trained triage model (step 15) must be retrained" if c["changed_split"] else "→ unchanged, a trained triage model stays valid")

In [ ]:
# Look at duplicates and borderline pairs: are they really the same photograph?
from PIL import Image as PILImage
pairs = pd.read_csv(PROJECT / "reports/m4_data/duplicate_pairs.csv", dtype=str)
cross = pairs[(pairs.a_dataset + pairs.a_part) != (pairs.b_dataset + pairs.b_part)]
show = pd.concat([pairs[pairs.kind == "review"].head(3), cross.head(2),
                  pairs[pairs.kind == "duplicate"].head(2)]).drop_duplicates().head(6)
if show.empty:
    print("No duplicate or borderline pairs found.")
for _, r in show.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
    for ax, side in zip(axes, ("a", "b")):
        im = PILImage.open(PROJECT / "data" / r[f"{side}_rel_path"])
        im.thumbnail((400, 400))
        ax.imshow(im); ax.set_axis_off()
        ax.set_title(f"{r[side + '_dataset']}/{r[side + '_part']} {r[side + '_id']}\ngrade {r[side + '_grade']}", fontsize=8)
    fig.suptitle(f"{r['kind']} · similarity {float(r['similarity']):.3f}", fontsize=9)
    plt.tight_layout(); plt.show()

**How to read it**
- **Splits:** APTOS 70 / 15 / 15 and IDRiD-grading train 85 / val 15 are stratified by grade; IDRiD's official test images stay in test; DRIVE and RITE get 5 cross-validation folds (they share photographs, so each photo keeps one fold).
- **Duplicates:** copies of a photo are grouped. If any copy is an official test image, every copy goes to test. Copies with *different* grades are excluded and listed.
- **Pairs shown above:** "duplicate" pairs were grouped automatically; "review" pairs were not — if a review pair is clearly the same photo, tell me and we lower the threshold.
- The **manifest fingerprint** identifies this exact split; quote it in your thesis so results are reproducible.

## 12 · Milestone 5 — vessel and artery/vein segmentation (T4 GPU)
A U-Net learns to draw the vessel map from a photo, and — with RITE — to label each vessel pixel as artery or vein. Separate artery and vein graphs stop the two trees from being joined at crossings (the M3 problem).

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save, then Run all (or steps 1–4 and this section). Free GPU time is limited, so start when you can keep the tab open.

What gets trained (about 10–30 min on a T4; progress is printed):
- **Cross-validation:** 5 models, each tested on the 4 annotated DRIVE photos it never saw (folds fixed per photo).
- **Final model:** trained on all 20 annotated photos, saved as `checkpoints/m5/final.pt` for later milestones. With RITE, it is also tested on the 20 DRIVE test photos.

Checkpoints are saved to Drive while training, so after a disconnect just re-run: it resumes.

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)
else:
    print("No GPU in this runtime. Runtime → Change runtime type → T4 GPU → Save, then re-run steps 1–4.")
    print("(Training on CPU works but takes hours; use PRESET = 'quick' below only to try the pipeline.)")
for name in ("DRIVE", "RITE"):
    unzip_all(PROJECT / "data" / name)
from drpinn.datasets.rite import find_rite
n_drive = len(find_drive_samples(PROJECT / "data" / "DRIVE")) if (PROJECT / "data" / "DRIVE").exists() else 0
n_rite = len(find_rite(PROJECT / "data" / "RITE").records) if (PROJECT / "data" / "RITE").exists() else 0
print(f"DRIVE photos with expert vessel masks: {n_drive} | RITE photos with artery/vein labels: {n_rite}")
if n_rite == 0:
    print("Without RITE the model learns vessels only; artery/vein separation needs RITE (step 11c).")

In [ ]:
# @title Train and evaluate { display-mode: "form" }
PRESET = "full"  # @param ["full", "quick"]
MODE = "both"    # @param ["both", "cv", "final"]
if n_drive == 0:
    print("DRIVE not found — complete step 10b first.")
elif PRESET == "full" and not torch.cuda.is_available():
    print("Stopped: PRESET 'full' needs a GPU — on Colab's CPU it would take about 30 hours.")
    print("Switch to a T4 runtime (Runtime → Change runtime type), or choose PRESET 'quick' (~1 hour on CPU, pipeline check only).")
else:
    run_live("scripts/m5_train_segmentation.py", "--drive-dir", PROJECT / "data/DRIVE",
             "--rite-dir", PROJECT / "data/RITE", "--out", PROJECT / f"reports/m5_{PRESET}",
             "--ckpt-dir", PROJECT / f"checkpoints/m5_{PRESET}", "--preset", PRESET, "--mode", MODE)

### 12b · Results

In [ ]:
res_dir = PROJECT / f"reports/m5_{PRESET}"
res = json.loads((res_dir / "m5_results.json").read_text()) if (res_dir / "m5_results.json").exists() else None
keys = [("dice", "Dice"), ("iou", "IoU"), ("sensitivity", "Sensitivity"), ("specificity", "Specificity"),
        ("auc", "AUC"), ("centreline_recall", "Centreline recall"), ("centreline_precision", "Centreline precision"),
        ("radius_median_abs_error", "Radius error (median)"), ("av_accuracy", "A/V accuracy"),
        ("av_sensitivity", "Artery sensitivity"), ("av_specificity", "Vein specificity")]
table = {}
if res is None:
    print(f"No '{PRESET}' results yet — run the training cell above first ('full' needs a T4 GPU runtime).")
for part, label in (() if res is None else (("cv", "Cross-validation (DRIVE 1st observer)"), ("test", "Test (DRIVE test photos, 2nd observer)"))):
    s = res.get(part, {}).get("summary", {})
    if s:
        table[label] = {name: f"{s[k]['mean']:.3f} ± {s[k]['sd']:.3f}" for k, name in keys if k in s}
if res is not None:
    display(pd.DataFrame(table))
    print("Device:", res["environment"]["device"], res["environment"].get("gpu") or "", "| artery/vein trained:", res["use_av"])
if res is not None and not res.get("test", {}).get("summary"):
    print("No test-photo results:", res.get("test", {}).get("note", "final model not trained yet"))

In [ ]:
# Per-photo graph check: does artery/vein separation remove the false crossings?
rows = []
for part in ("cv", "test"):
    for r in (res or {}).get(part, {}).get("per_image", []):
        if "combined_components" in r:
            rows.append({"set": part, "photo": r["image_id"],
                         "combined: crossings": r["combined_crossings_split"] + r["combined_possible_crossings"],
                         "combined: pieces": r["combined_components"],
                         "artery: crossings": r.get("artery_crossings_split", 0) + r.get("artery_possible_crossings", 0),
                         "artery: pieces": r.get("artery_components"),
                         "vein: crossings": r.get("vein_crossings_split", 0) + r.get("vein_possible_crossings", 0),
                         "vein: pieces": r.get("vein_components")})
if rows:
    g = pd.DataFrame(rows)
    display(g)
    print("Totals — crossings in combined graphs:", int(g["combined: crossings"].sum()),
          "| in artery graphs:", int(g["artery: crossings"].sum()), "| in vein graphs:", int(g["vein: crossings"].sum()))
elif res is not None:
    print("Artery/vein graphs need RITE (step 11c).")
for png in sorted((res_dir / "overlays").glob("*.png"))[:4]:
    display(ShowImage(filename=str(png)))

**How to read it**
- **Dice / IoU / sensitivity / specificity / AUC** — pixel agreement with the expert vessel mask, inside the field of view, at a 0.5 threshold (no tuning on test photos).
- **Centreline recall / precision** — how much of the expert's vessel network the model's graph finds, and how much of the model's graph is real (within 2 px).
- **Radius error** — the number the physics cares about (M2: about ≤ 5 % keeps the high-stress ranking stable). It is measured against the width estimated from the expert mask, which itself carries ~1–3 % error on thin vessels (M3).
- **A/V accuracy** — artery vs vein on the expert's vessel centreline, skipping crossings and uncertain pixels.
- **Crossings / pieces** — the combined graph should show crossings; the artery and vein graphs should show far fewer and be mostly one piece each.
- **Protocol note for your thesis:** DRIVE's official test masks (1st observer) are withheld; the test row uses the 2nd observer's masks via RITE, so it is not directly comparable to papers that used the 1st observer.

## 13 · Milestone 8 — does a PINN add value over the classical solver? (CPU — no GPU needed)
Done before Milestones 6–7 because it needs neither the GPU nor IDRiD/APTOS. This is the core question behind your title (plan assumption A11), answered on identical networks and boundary conditions:

| Method | What it is |
|---|---|
| direct solve | the classical sparse solver from Milestone 1 — the reference |
| Jacobi(30), CG(30) | classical iterative solvers allowed 30 steps |
| PI-GNN(30) | a physics-informed graph neural network: trained **once** on random synthetic trees using only the physics residual (no solver answers), then 30 learned steps per network |
| nodal PINN | per network: pressures as trainable numbers, minimise the physics residual |
| coordinate PINN | per network: the classic PINN — a neural network of each node's (x, y) position; its Fourier scale is picked on separate tuning networks, never on the test set |

Test networks: 20 random synthetic trees plus up to 10 graphs from your DRIVE expert masks. **quick** ≈ 5–10 min, **full** ≈ 30–40 min on Colab's CPU (estimates; progress is printed).

In [ ]:
# @title Run the comparison { display-mode: "form" }
M8_MODE = "quick"  # @param ["quick", "full"]
args = ["scripts/m8_pinn_vs_solver.py", "--drive-dir", PROJECT / "data/DRIVE", "--out", PROJECT / f"reports/m8_{M8_MODE}"]
run_live(*(args + (["--quick"] if M8_MODE == "quick" else [])))

In [ ]:
m8 = json.loads((PROJECT / f"reports/m8_{M8_MODE}/m8_results.json").read_text())
for st, methods in m8["summary"].items():
    n = next(iter(methods.values()))["networks"]
    print(f"{st} networks (n = {n})")
    display(pd.DataFrame({m: {"pressure error, median": f"{s['pressure_error_median']:.1e}",
                              "pressure error, worst": f"{s['pressure_error_worst']:.1e}",
                              "WSS top-10% kept": f"{s['wss_top10_overlap_median']:.2f}",
                              "WSS error, median": f"{s['wss_median_rel_error']:.1e}",
                              "time per network": f"{s['seconds_median'] * 1000:.1f} ms"}
                          for m, s in methods.items()}).T)
print(f"PI-GNN training (once): {m8['gnn_training']['seconds']:.0f} s | coordinate-PINN Fourier scale chosen: "
      f"{m8['settings']['coordinate_sigma']} | total {m8['runtime_s'] / 60:.1f} min")
display(ShowImage(filename=str(PROJECT / f"reports/m8_{M8_MODE}/m8_accuracy_vs_time.png")))

**How to read it** (pressure errors are fractions of the inlet–outlet pressure drop; 1e-2 = 1 %)
- The **direct solve** is exact and takes a few milliseconds. Every other method is judged against it.
- **Per-network PINNs** (nodal, coordinate) are trained from scratch for every network, so they take seconds to minutes for an answer the direct solve gives exactly in milliseconds. The classic coordinate PINN also gets worse as networks grow.
- The **PI-GNN** is the interesting case: for the same 30 steps it is usually far more accurate than Jacobi and CG — a genuine learned speed-up of iterative solving — but its worst cases can be large, and it is still slower and less accurate than the direct solve.
- **Right-hand plot:** the physics loss tracks the error, but at the same loss the error can differ about 100-fold — a small PINN loss alone does not prove the pressures are right.
- **What this means for your thesis:** for the *forward* problem (known geometry and boundary conditions), a PINN does not add value over the classical solver here; report this as a result, not a failure (PDF §9.4). Where physics-informed learning could add value is in *inverse* problems — inferring unknown parameters from measurements — which ordinary fundus photos do not provide. Discuss the title's framing with your supervisor.

## 14 · Milestone 7 — the core experiment: do high-shear vessel regions sit where the lesions are? (CPU is fine)
The question was fixed in advance in `docs/m7_analysis_plan.md` (pre-registered 31 Oct 2026, before any real result): **within one eye, do regions whose vessels the physics ranks as high wall-shear stress contain expert-marked red lesions (microaneurysms + haemorrhages) more often than other vessel regions?**

For each of the 81 IDRiD segmentation photos:
1. The M5 model draws vessels, arteries and veins, giving separate artery and vein graphs.
2. The solver runs 50 times per tree, sampling the unknowns each time: outlet resistance, outlet pressure, radius error, viscosity model and pixel size.
3. Each vessel segment gets P(top 10 % shear).
4. Each 32 px patch takes the maximum over its segments, and the patches are compared with the lesion masks.

**No lesion information is used to build the vessels or run the physics.**

**Primary result:** the mean within-image AUROC over the 27 official test photos, with a 95 % t-interval. Beating 0.5 is not enough on its own, so the table also compares it photo by photo with simple baselines: vessel density, distance to the fovea, vessel thinness and the ranking without uncertainty.

**Needs:** the IDRiD segmentation part (step 11a) and the M5 model (step 12, preset `full`). **Runtime:** roughly 15–40 min on CPU; each photo's progress is printed.

This is cross-sectional: it cannot show that shear *predicts future* lesions.

In [ ]:
# @title Check what M7 needs { display-mode: "form" }
M5_MODEL = "full"  # @param ["full", "quick"]
from drpinn.datasets.idrid import find_idrid
IDRID_DIR = PROJECT / "data" / "IDRiD"
if IDRID_DIR.exists():
    unzip_all(IDRID_DIR)
seg = find_idrid(IDRID_DIR).segmentation if IDRID_DIR.exists() else []
with_od = [r for r in seg if "OD" in r.extra.get("masks", {})]
print(f"IDRiD segmentation photos: {len(seg)} | with optic-disc mask: {len(with_od)} "
      f"| official test: {sum(r.official_split == 'test' for r in with_od)}")
M5_CKPT = PROJECT / f"checkpoints/m5_{M5_MODEL}/final.pt"
print("M5 model:", M5_CKPT.relative_to(PROJECT), "→", "found" if M5_CKPT.exists() else "NOT FOUND (finish step 12 first)")
if M5_MODEL == "quick":
    print("Note: the 'quick' model only checks the pipeline. Thesis results must use the 'full' model.")
m5_json = PROJECT / f"reports/m5_{M5_MODEL}/m5_results.json"
cv = json.loads(m5_json.read_text()).get("cv", {}).get("summary", {}) if m5_json.exists() else {}
if "radius_median_abs_error" in cv:
    RADIUS_SIGMA = round(cv["radius_median_abs_error"]["mean"] / 0.674, 3)
    print(f"Radius error in M5 cross-validation: median {cv['radius_median_abs_error']['mean']:.3f} → σ = {RADIUS_SIGMA} (plan rule)")
    if RADIUS_SIGMA > 0.3:
        print("Warning: this radius error is large, so the sampled rankings will be mostly noise. Check the M5 results "
              "(step 12b) first; the 'full' model should do much better.")
else:
    RADIUS_SIGMA = 0.10
    print("No M5 cross-validation result → σ = 0.10 (plan default)")
M7_READY = bool(with_od) and M5_CKPT.exists()
if not with_od:
    print("IDRiD segmentation images not found: upload 'A. Segmentation.zip' to dr-pinn/data/IDRiD (step 11a).")

In [ ]:
# @title Run the experiment (settings fixed by the pre-registered plan) { display-mode: "form" }
if not M7_READY:
    print("Not ready: see the check above.")
else:
    m7_out = PROJECT / f"reports/m7_{M5_MODEL}"
    if (m7_out / "m7_results.json").exists():  # results from other code are kept aside, never mixed or overwritten
        _v = json.loads((m7_out / "m7_results.json").read_text()).get("drpinn_version")
        if _v != drpinn.__version__:
            _dest = m7_out.with_name(f"{m7_out.name}_previous_code_{_v}")
            shutil.move(str(m7_out), str(_dest))
            print(f"Earlier results were made by code {_v}: moved to reports/{_dest.name}; running fresh with {drpinn.__version__}.")
    run_live("scripts/m7_lesion_association.py", "--idrid-dir", IDRID_DIR, "--model", M5_CKPT,
             "--out", m7_out, "--radius-sigma", RADIUS_SIGMA)

### 14b · Results

In [ ]:
m7_dir = PROJECT / f"reports/m7_{M5_MODEL}"
if not (m7_dir / "m7_results.json").exists():
    print("No M7 results yet: run the cell above.")
else:
    m7 = json.loads((m7_dir / "m7_results.json").read_text())
    p = m7["primary"]
    print(f"{m7['images']} photos ({m7['test_images']} official test), {m7['patches']} vessel patches, "
          f"radius σ {m7['config']['mc']['radius_sigma']}, {m7['config']['mc']['n_samples']} samples per tree")
    if p["available"]:
        print(f"\nPRIMARY: within-image AUROC {p['within_image_auroc']:.3f} [{p['ci'][0]:.3f}, {p['ci'][1]:.3f}], "
              f"p = {p['p_vs_0.5']:.3f} vs 0.5, {p['images']} test photos → {p['verdict']}\n")
        display(pd.DataFrame({b: {"high-shear minus baseline (AUROC)": f"{d['difference']:+.3f}",
                                  "95 % CI": f"[{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]",
                                  "p": f"{d['p_value']:.3f}", "photos": d["images"]}
                              for b, d in p["vs_baselines"].items()}).T)
    cm = m7["results"].get("combined_models", {})
    if "physics added (paired)" in cm:
        d = cm["physics added (paired)"]
        print(f"Exploratory: adding physics to position + geometry changes AUROC by {d['difference']:+.3f} "
              f"[{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]")
    print("Median rank stability (artery / vein):", {k: round(v, 2) for k, v in m7["median_rank_stability"].items()})
    print("Median share of red-lesion pixels drawn as vessel:", m7["red_lesion_pixels_labelled_vessel_median"])
    display(ShowImage(filename=str(m7_dir / "m7_forest.png")))
    for png in sorted(m7_dir.glob("m7_example_*.png"))[:2]:
        display(ShowImage(filename=str(png)))

**How to read it**
- **AUROC 0.5** means high-shear regions are no more likely to hold red lesions than other vessel regions. Above 0.5 means more likely; below 0.5 means less likely, i.e. lesions sit where shear is *not* high (the opposite hypothesis).
- **The verdict follows the rule written before the run:**
  - CI entirely above 0.5: positive association;
  - CI includes 0.5: no evidence;
  - CI entirely below 0.5: negative association.
  
  Any of these is a valid thesis result; report it as it is.
- **Baselines table:** a positive difference means the physics score ranks lesion regions better than that simple baseline in the same photos. If the physics beats 0.5 but not vessel density or position, the association is explained by *where vessels are*, not by the physics.
- **Rank stability** near 1 means the ranking barely moves when the unknowns change; around 0.3 means the unknowns dominate (as in the synthetic checks).
- **Lesion pixels drawn as vessel:** haemorrhages can be mistaken for vessels. A high share means lesions contaminate the geometry, so mention it as a limitation.
- Everything except the primary row is secondary or exploratory (not corrected for multiple comparisons).

## 15 · Milestone 6 — DR grade triage model (T4 GPU)
A MobileNetV3 learns the ICDR grade (0–4) from the APTOS photos. Everything that decides how it is used is fixed on the **validation** photos before any test photo is scored (`docs/m6_triage.md`, pre-registered):
- the training epoch;
- a calibration temperature;
- a referral threshold giving ≥ 90 % sensitivity for referable DR (grade ≥ 2);
- an uncertainty limit (the 10 % least certain photos are marked "needs a human grader").

The model is then tested on held-out APTOS photos (internal) and on **IDRiD (external: another clinic and camera, never used for training)**. Each photo also gets image-quality flags and a Grad-CAM map.

**Needs:**
- the APTOS working copy (step 11b);
- the splits rebuilt **after** APTOS was added (step 11d);
- IDRiD 'B. Disease Grading' for the external test (step 11a).

**Runtime (estimates):**
- Reading the ~4,000 photos from Drive and caching them at 512 px on Colab's disk takes about 15–30 min. A new session repeats this.
- Training needs the T4 GPU: `full` takes about 10 min, `quick` about 3 min. Checkpoints go to Drive after every epoch, so after a disconnect just re-run; training resumes.
- **Once the model is trained, re-running this step only re-evaluates it, and that works on CPU.** It skips training, and after the first evaluation it no longer reads the training photos.

In [ ]:
import csv as _csv
from collections import Counter
man = PROJECT / "data/splits/manifest.csv"
counts = Counter()
if man.exists():
    with open(man) as f:
        for r in _csv.DictReader(f):
            if r["part"] == "grading" and r["dr_grade"] != "" and r.get("label_conflict", "0") != "1":
                counts[(r["dataset"], r["split"])] += 1
print("Graded photos in the manifest:", dict(counts) or "none (run steps 11a, 11b and 11d)")
M6_READY = counts[("APTOS2019", "train")] > 0 and counts[("APTOS2019", "val")] > 0
if not M6_READY:
    print("No APTOS train/val photos: finish step 11b (APTOS working copy), then re-run step 11d.")
if not any(k[0] == "IDRiD" for k in counts):
    print("No IDRiD grading photos: the external test will be skipped (upload 'B. Disease Grading.zip', step 11a, then 11d).")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# @title Train (GPU) or re-evaluate (CPU is fine), calibrate and test { display-mode: "form" }
M6_PRESET = "full"  # @param ["full", "quick"]
_last = PROJECT / f"checkpoints/m6_{M6_PRESET}/last.pt"
M6_TRAINED = _last.exists() and torch.load(_last, map_location="cpu", weights_only=False)["state"].get("complete", False)
if not M6_READY:
    print("Not ready: see the check above.")
elif not torch.cuda.is_available() and not M6_TRAINED:
    print("Stopped: training needs a GPU (on Colab's CPU even 'quick' would take many hours).")
    print("Runtime → Change runtime type → T4 GPU → Save, then re-run steps 1–4 and this section.")
else:
    if M6_TRAINED:
        print("Model already trained: evaluating only (no GPU needed).")
    run_live("scripts/m6_triage.py", "--data-dir", PROJECT / "data", "--preset", M6_PRESET,
             "--out", PROJECT / f"reports/m6_{M6_PRESET}", "--ckpt-dir", PROJECT / f"checkpoints/m6_{M6_PRESET}",
             "--cache-dir", "/content/m6_cache", "--workers", "6")

### 15b · Results

In [ ]:
m6_dir = PROJECT / f"reports/m6_{M6_PRESET}"
if not (m6_dir / "m6_results.json").exists():
    print("No M6 results yet: run the cell above (needs a T4 GPU runtime).")
else:
    m6 = json.loads((m6_dir / "m6_results.json").read_text())
    print("\n".join(m6["headline"]))
    op = m6["operating_point"]
    print(f"\nFixed on validation: epoch {m6['training']['best_epoch']} | temperature {m6['temperature']:.2f} | "
          f"referral threshold P(referable) ≥ {op['referral_threshold']:.3f} | uncertainty limit {op['uncertainty_limit']:.2f}")
    def ci(m):
        return f"{m['value']:.3f} [{m['ci'][0]:.3f}, {m['ci'][1]:.3f}]"
    table = {}
    for name, s in m6["sets"].items():
        r, c = s["referable"], s["calibration"]
        table[s["description"]] = {
            "photos": s["images"], "QWK": ci(s["qwk"]), "accuracy": ci(s["accuracy"]), "macro-F1": ci(s["macro_f1"]),
            "referable AUC": ci(r["auc"]), "sensitivity": ci(r["sensitivity"]), "specificity": ci(r["specificity"]),
            "referable ECE before → after": f"{c['referable_ece_before']:.3f} → {c['referable_ece_after']:.3f}",
            "quality-flagged photos": s["quality"]["quality_flagged"]["images"],
            "marked uncertain": f"{s['uncertain_share']:.0%}"}
    display(pd.DataFrame(table))
    ext = next((n for n in ("external_idrid_test", "test_internal") if n in m6["sets"]), None)
    if ext:
        print(f"Setting aside the least certain photos ({m6['sets'][ext]['description']}):")
        display(pd.DataFrame(m6["sets"][ext]["selective"]).set_index("coverage").round(3))
    for f in m6["figures"]:
        display(ShowImage(filename=str(m6_dir / f)))
    print("Example triage card:", json.dumps(m6["example_cards"][0], indent=1)[:900])

**How to read it**
- **QWK** (quadratic weighted kappa) is agreement with the graders on the 0–4 scale: 1 is perfect, 0 is chance, and larger disagreements cost more.
- **Referable AUC** asks how well P(grade ≥ 2) separates referable from non-referable photos, at any threshold.
- **Sensitivity / specificity** use the threshold fixed on validation for ≥ 90 % sensitivity. If sensitivity on IDRiD drops well below 90 %, the threshold did not carry over to the other camera. That is a real finding about deployment, not a bug.
- **ECE** (calibration error) should fall after temperature scaling on APTOS. It may not on IDRiD, because calibration does not carry over to another camera either.
- **Setting aside uncertain photos:** if accuracy rises as coverage falls, the uncertainty score is useful for sending hard cases to a human grader.
- **Quality-flagged photos** fall outside the range of the training photos. Expect worse results on them.
- **Grad-CAM** shows where the evidence came from, roughly. It is not a lesion detector, and the "missed" and "false alarm" examples are as important as the correct ones.
- Research prototype: these are not clinical referral decisions, and no follow-up interval is given.

## 16 · Milestones 9–10 — the research dashboard (CPU is fine)
This step starts the analysis service and opens a dashboard in a new browser tab. Upload any fundus photo and the dashboard shows:
- image quality;
- the DR grade triage card and its Grad-CAM;
- the vessel, artery and vein map;
- the Monte Carlo wall-shear-stress risk map, with every assumption behind it.

**Click the optic disc** on the uploaded photo before analysing; otherwise it is estimated.

It uses the models you trained: steps 12 (vessels) and 15 (triage). Any part whose model is missing says so and names the step that makes it. One analysis takes roughly 10–60 s on CPU.

The link works only while this notebook is running, and only for your Google account. **In Firefox the link and the embedded view are often blocked** (tracking protection): use **step 16c** instead, which shows the same dashboard inside the notebook. Research prototype: the page says so on every screen.

In [ ]:
# @title Start the API and the dashboard { display-mode: "form" }
import socket, urllib.request
from drpinn.api.colab import serve_in_background

def _health(port, timeout=5):
    try:
        return json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/api/health", timeout=timeout).read())
    except Exception:
        return None

PORT, health = None, None
for _p in range(8000, 8011):  # reuse a dashboard server with this code version, else start one on a free port
    _h = _health(_p)
    if _h and _h.get("version") == drpinn.__version__:
        PORT, health = _p, _h
        print(f"A dashboard server is already running on port {_p}: reusing it.")
        break
if PORT is None:
    for _p in range(8000, 8011):
        with socket.socket() as _s:
            if _s.connect_ex(("127.0.0.1", _p)) != 0:
                PORT = _p
                break
    SERVER = serve_in_background(PROJECT, port=PORT)
    print(f"Server started on port {PORT}. Loading the models from Drive (the first time can take a minute) …", flush=True)
    health = _health(PORT, timeout=600)
if health is None:
    print("The server is not answering. Run this cell once more; if it still fails, use step 16c (no server needed).")
else:
    display(pd.DataFrame({k: {"available": v["available"], "detail": v.get("detail", "")}
                          for k, v in health["components"].items()}).T)
if health is not None and IN_COLAB:
    from google.colab import output
    print("The dashboard is shown below (scroll inside it). To open it in its own tab instead, use this link:")
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the DR-PINN dashboard in a new tab")
    output.serve_kernel_port_as_iframe(PORT, height=1300)
    print("If the dashboard stays blank or shows an error, use step 16c below: same analysis, no browser link needed.")
elif health is not None:
    print(f"Dashboard: http://127.0.0.1:{PORT}")

### 16b · How good is the optic-disc estimate? (IDRiD, CPU, ~2 min)
The dashboard estimates the disc when you don't click it. This cell measures that estimate on the 81 IDRiD photos with expert disc masks:
- how often the estimated centre falls inside the expert disc;
- how far off it is;
- the real disc-to-field-of-view ratio (the estimate assumes 0.16, measured on IDRiD).

It reuses the vessel maps cached by step 14, if that has run. Quote these numbers whenever you describe the dashboard.

In [ ]:
run_live("scripts/m9_check_disc.py", "--idrid-dir", PROJECT / "data/IDRiD",
         "--probs", PROJECT / "reports/m7_full/probs", "--out", PROJECT / "reports/m9_disc")

### 16c · The dashboard for one photo, inside the notebook (works in Firefox, no link needed)
Runs the same analysis as the live dashboard and shows the **same 10-section dashboard** under this cell. It also saves it as one HTML file (`reports/m9_examples/<photo>_dashboard.html`) that opens in any browser, even offline, so you can keep it or send it to your supervisor.
- **`PHOTO`:** any fundus photo in your Drive (Files panel on the left → right-click the photo → Copy path). Leave it empty to use the first IDRiD test photo.
- **`DISC_X`, `DISC_Y`:** the optic-disc centre in photo pixels, if you want to set it yourself. Leave them at -1 and the disc is estimated (it hit the real disc in 99 % of IDRiD photos, step 16b).
- **`DOWNLOAD`:** tick it to also download the dashboard file to your computer.

In [ ]:
# @title Dashboard for one photo { display-mode: "form" }
PHOTO = ""  # @param {type:"string"}
DISC_X = -1  # @param {type:"number"}
DISC_Y = -1  # @param {type:"number"}
DOWNLOAD = False  # @param {type:"boolean"}
from IPython.display import HTML
from PIL import Image as PILImage
from drpinn.api.service import Analyser, ServiceConfig
from drpinn.api.report import save_static_dashboard
if not PHOTO:
    _c = sorted((PROJECT / "data/IDRiD").rglob("IDRiD_55.jp*g")) or sorted((PROJECT / "data/IDRiD").rglob("IDRiD_*.jp*g"))
    PHOTO = str(_c[0]) if _c else ""
if not PHOTO or not Path(PHOTO).exists():
    print("Set PHOTO to the path of a fundus photo in your Drive (Files panel → right-click → Copy path).")
else:
    if "ANALYSER" not in globals():
        print("Loading the models (first time only) …", flush=True)
        ANALYSER = Analyser(ServiceConfig(project=PROJECT))
    rgb = np.asarray(PILImage.open(PHOTO).convert("RGB"))
    disc = (float(DISC_X), float(DISC_Y), None) if DISC_X >= 0 and DISC_Y >= 0 else None
    print(f"Analysing {Path(PHOTO).name} ({rgb.shape[1]} x {rgb.shape[0]} px) … about 10–60 s", flush=True)
    R = ANALYSER.analyse(rgb, disc)
    out = save_static_dashboard(PROJECT / "reports/m9_examples" / f"{Path(PHOTO).stem}_dashboard.html",
                                R, ANALYSER.health(), rgb, Path(PHOTO).name)
    (out.with_suffix(".json")).write_text(json.dumps(R))
    print(f"Done in {R['seconds']['total']:.0f} s. Saved {out.relative_to(PROJECT)} (open it in any browser).")
    display(HTML(out.read_text(encoding="utf-8")))
    if DOWNLOAD and IN_COLAB:
        from google.colab import files
        files.download(str(out))

## 17 · Milestone 11 — every result in one summary
Collects the results of every milestone from `reports/` into one machine-readable file (`reports/summary/results_summary.json`) and one readable report (`results_summary.md`). Each number is copied, never recomputed, and the report names the file it came from. It also warns about anything that must not go into the thesis:
- synthetic software checks;
- 'quick' pipeline runs;
- M6 IDRiD numbers made before the 0.7.2 fix.

Re-run it whenever a result changes. The thesis structure, the claims to avoid and the full limitations list are in `docs/thesis_outline.md`.

In [ ]:
from IPython.display import Markdown
run("scripts/m11_collect_results.py", "--project", PROJECT)
display(Markdown((PROJECT / "reports/summary/results_summary.md").read_text()))

## What's next
Everything is built. What remains is running it on your data, in this order:
1. **Step 11d** (rebuild the splits with the IDRiD fix), then **step 15** (re-evaluates your M6 model; CPU is fine).
2. **Step 14** (M7, the core experiment) with your `full` vessel model.
3. **Step 13** with `M8_MODE = "full"` (CPU, ~30–40 min).
4. **Step 16b** (disc check), then **step 17** (the summary).

Send me `reports/summary/results_summary.md` from your Drive. I'll check every number, then we write the results and discussion chapters around it, following `docs/thesis_outline.md`.